In [ ]:
!pip install ultralytics

import os
import shutil
import torch
from google.colab import files
from ultralytics import YOLO

# Setup device
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Clean previous extractions
dataset_dir = './parking_space_dataset'
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)

# Locate zip file
zip_path = None
for f in os.listdir('/content'):
    if f.endswith('.zip') or 'parking' in f.lower():
        full_path = os.path.join('/content', f)
        if os.path.isfile(full_path):
            zip_path = full_path
            break

if not zip_path:
    raise FileNotFoundError("Dataset zip file not found in /content. Please upload it first.")

print(f"Extracting {zip_path}...")
!unzip -q -o "{zip_path}" -d {dataset_dir}

# Find data.yaml
yaml_path = None
for root, _, files_list in os.walk(dataset_dir):
    for f in files_list:
        if f.lower() in ["data.yaml", "data.yml"]:
            yaml_path = os.path.abspath(os.path.join(root, f))
            break

if not yaml_path:
    raise FileNotFoundError("Could not find data.yaml inside the extracted archive.")

print(f"Found config at: {yaml_path}")

# Initialize and train model
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data=yaml_path,
    epochs=30,
    imgsz=640,
    device=device,
    project="SmartParking",
    name="YOLOv8_Spaces_Seg"
)

# Download trained weights
weights = "/content/SmartParking/YOLOv8_Spaces_Seg/weights/best.pt"
if os.path.exists(weights):
    print("Training complete. Downloading weights...")
    files.download(weights)
else:
    print("Warning: Weights file not found at the expected path.")